[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astheeggeggs/testing_colab/blob/main/LDSC_Practical_2_Colab.ipynb)

# LDSC Practical 2 — Genetic Correlation (Colab version)

**MadhurBain Singh, Benjamin Neale**  
**March 5th, 2025**  
**Adapted from Andrew Grotzinger, March 2023**

This notebook is a Colab-friendly rewrite of the original R practical.
It assumes an **R runtime** in Colab.

**Before you start**
- Put the practical files in one public Google Drive folder, or upload them into `/content`.
- Make sure the folder contains the `EUR/` and `EAS/` directories used below.
- Run the setup cells from top to bottom.

## 1) Download the practical files into Colab

Replace `PASTE_PUBLIC_DRIVE_FOLDER_LINK_HERE` with the public folder link that contains the practical files.
If you already uploaded the files into the runtime, you can skip this cell and set `root_dir` below manually.

In [ ]:
# Install gdown so Colab can fetch a public Google Drive folder
system('python -m pip -q install gdown')

# Replace this with your public Google Drive folder link
drive_folder_url <- "PASTE_PUBLIC_DRIVE_FOLDER_LINK_HERE"

if (grepl("PASTE_PUBLIC_DRIVE_FOLDER_LINK_HERE", drive_folder_url, fixed = TRUE)) {
  message("Replace drive_folder_url with your public Google Drive folder link, then rerun this cell.")
} else {
  system(sprintf('gdown --folder "%s" -O /content/LDSC_Practical_2', drive_folder_url))
  system('find /content/LDSC_Practical_2 -maxdepth 3 -type f | head -50')
}

## 2) Environment setup / required packages

This notebook uses an R runtime.
We install the packages needed for the practical from CRAN and GitHub.

Packages used here:
- `data.table`
- `dplyr`
- `remotes`
- `GenomicSEM`
- `corrplot`

In [ ]:
# System libraries that are often useful for R package installation in Colab
system("apt-get update -qq")
system("apt-get install -y -qq libcurl4-openssl-dev libxml2-dev libssl-dev")

In [ ]:
# Install CRAN packages
install.packages(
  c("data.table", "dplyr", "remotes", "corrplot"),
  repos = "https://cloud.r-project.org"
)

# Install GenomicSEM from GitHub
remotes::install_github("GenomicSEM/GenomicSEM", upgrade = "never")

### Load packages

In [ ]:
library(data.table)
library(dplyr)
library(GenomicSEM)
library(corrplot)

sessionInfo()

## 3) Colab working directory and file helpers

The code below assumes that the practical files are available in `/content/LDSC_Practical_2`.
If you used the download cell above, that is where the files will be.

In [ ]:
root_dir <- "/content/LDSC_Practical_2"

if (!dir.exists(root_dir)) {
  dir.create(root_dir, recursive = TRUE, showWarnings = FALSE)
}

# Keep paths explicit rather than relying on setwd()
eur_dir <- file.path(root_dir, "EUR")
eas_dir <- file.path(root_dir, "EAS")

# Helper functions for cleaner file previews in notebooks
preview_text <- function(path, n = 10) {
  cat(readLines(path, n = n), sep = "\n")
  cat("\n")
}

preview_gz <- function(path, n = 10) {
  con <- gzfile(path, open = "rt")
  on.exit(close(con), add = TRUE)
  cat(readLines(con, n = n), sep = "\n")
  cat("\n")
}

count_lines <- function(path) {
  length(readLines(path, warn = FALSE))
}

count_gz_lines <- function(path) {
  con <- gzfile(path, open = "rt")
  on.exit(close(con), add = TRUE)
  length(readLines(con, warn = FALSE))
}

list.files(root_dir, recursive = TRUE)[1:min(20, length(list.files(root_dir, recursive = TRUE)))]

## Part 1: Continuous traits example in European ancestry

This section uses chromosome 1 summary statistics for BMI and height.

### HapMap3 SNPs

As before, we restrict the analyses to high-quality SNPs in the GWAS summary statistics that overlap with the pre-computed LD scores.

In [ ]:
hm3_eur <- file.path(eur_dir, "eur_w_ld_chr", "w_hm3.snplist")

# Take a look at the first few lines
preview_text(hm3_eur)

# Count the number of lines
cat("Number of lines in HM3 file:", count_lines(hm3_eur), "\n")

**Question:** How many SNPs are there in the HapMap3 reference file?

### Step 1: Munge the data

In [ ]:
# Look at the summary statistics files and their columns
preview_text(file.path(eur_dir, "GIANT_UKB_BMI_EUR_chr1.txt"))
preview_text(file.path(eur_dir, "Yengo_Height_EUR_chr1.txt"))

# For multivariate LDSC, specify a vector of file paths with GWAS summary statistics
files_eur <- c(
  file.path(eur_dir, "GIANT_UKB_BMI_EUR_chr1.txt"),
  file.path(eur_dir, "Yengo_Height_EUR_chr1.txt")
)

# Trait names should match the order of the file paths
trait_names_eur <- c("BMI_chr1", "Height_chr1")

# Quality filters
info.filter <- 0.9
maf.filter <- 0.01

munge(
  files = files_eur,
  hm3 = hm3_eur,
  trait.names = trait_names_eur,
  info.filter = info.filter,
  maf.filter = maf.filter
)

### Step 2: Run LDSC

In [ ]:
# Examine the munged summary statistics
preview_gz(file.path(eur_dir, "BMI.sumstats.gz"))
preview_gz(file.path(eur_dir, "Height.sumstats.gz"))

traits_eur <- c(
  file.path(eur_dir, "BMI.sumstats.gz"),
  file.path(eur_dir, "Height.sumstats.gz")
)

ld_eur <- file.path(eur_dir, "eur_w_ld_chr")
wld_eur <- file.path(eur_dir, "eur_w_ld_chr")
sample.prev_eur <- c(NA, NA)
population.prev_eur <- c(NA, NA)
trait_names_ldsc_eur <- c("BMI", "Height")

LDSC_EUR <- ldsc(
  traits = traits_eur,
  sample.prev = sample.prev_eur,
  population.prev = population.prev_eur,
  ld = ld_eur,
  wld = wld_eur,
  trait.names = trait_names_ldsc_eur
)

save(LDSC_EUR, file = file.path(eur_dir, "LDSC_EUR.RData"))

### Inspect the outputs

In [ ]:
# Genetic covariance matrix
LDSC_EUR$S

# Genetic correlation matrix
cov2cor(LDSC_EUR$S)

# Standard errors from the V matrix
k <- nrow(LDSC_EUR$S)
SE <- matrix(0, k, k)
SE[lower.tri(SE, diag = TRUE)] <- sqrt(diag(LDSC_EUR$V))
SE

# Z-statistics
Z <- LDSC_EUR$S / SE
Z

# P-values
P <- 2 * pnorm(Z, lower.tail = FALSE)
P

### Optional: Create an rg heatmap

In [ ]:
rownames(LDSC_EUR$S) <- colnames(LDSC_EUR$S)

corrplot(
  corr = cov2cor(LDSC_EUR$S),
  method = "color",
  addCoef.col = "dark grey",
  add = FALSE,
  bg = "white",
  diag = TRUE,
  outline = TRUE,
  mar = c(0, 0, 2, 0),
  number.cex = 2,
  cl.pos = "b",
  cl.ratio = 0.125,
  cl.align.text = "l",
  cl.offset = 0.2,
  tl.srt = 45,
  tl.pos = "lt",
  tl.offset = 0.2,
  tl.col = "black",
  pch.col = "white",
  addgrid.col = "black",
  xpd = TRUE,
  tl.cex = 1.3,
  is.corr = TRUE,
  title = "European"
)

## Part 2: Continuous traits example in East Asian ancestry

The summary statistics and LD scores for this section are from East Asian ancestry.

In [ ]:
# Look at the files
preview_text(file.path(eas_dir, "BMI_EAS_chr1.txt"))
preview_text(file.path(eas_dir, "Yengo_Height_EAS_chr1.txt"))

files_eas <- c(
  file.path(eas_dir, "BMI_EAS_chr1.txt"),
  file.path(eas_dir, "Yengo_Height_EAS_chr1.txt")
)

hm3_eas <- file.path(eas_dir, "eas_ldscores", "w_hm3.snplist")
preview_text(hm3_eas)
cat("Number of lines in EAS HM3 file:", count_lines(hm3_eas), "\n")

trait_names_eas <- c("BMI", "Height")
N_eas <- c(163835, NA)
info.filter <- 0.9
maf.filter <- 0.01

munge(
  files = files_eas,
  hm3 = hm3_eas,
  trait.names = trait_names_eas,
  N = N_eas,
  info.filter = info.filter,
  maf.filter = maf.filter
)

In [ ]:
preview_gz(file.path(eas_dir, "BMI.sumstats.gz"))
preview_gz(file.path(eas_dir, "Height.sumstats.gz"))

traits_eas <- c(
  file.path(eas_dir, "BMI.sumstats.gz"),
  file.path(eas_dir, "Height.sumstats.gz")
)

ld_eas <- file.path(eas_dir, "eas_ldscores")
wld_eas <- file.path(eas_dir, "eas_ldscores")
sample.prev_eas <- c(NA, NA)
population.prev_eas <- c(NA, NA)
trait_names_ldsc_eas <- c("BMI", "Height")

LDSC_EAS <- ldsc(
  traits = traits_eas,
  sample.prev = sample.prev_eas,
  population.prev = population.prev_eas,
  ld = ld_eas,
  wld = wld_eas,
  trait.names = trait_names_ldsc_eas
)

save(LDSC_EAS, file = file.path(eas_dir, "LDSC_EAS.RData"))

In [ ]:
# Genetic covariance matrix
LDSC_EAS$S

# Compare with EUR if needed
load(file.path(eur_dir, "LDSC_EUR.RData"))
LDSC_EUR$S

# Genetic correlation matrix
cov2cor(LDSC_EAS$S)

# Standard errors
k <- nrow(LDSC_EAS$S)
SE <- matrix(0, k, k)
SE[lower.tri(SE, diag = TRUE)] <- sqrt(diag(LDSC_EAS$V))
SE

# Z-statistics
Z <- LDSC_EAS$S / SE
Z

# P-values
P <- 2 * pnorm(Z, lower.tail = FALSE)
P

In [ ]:
rownames(LDSC_EAS$S) <- colnames(LDSC_EAS$S)

corrplot(
  corr = cov2cor(LDSC_EAS$S),
  method = "color",
  addCoef.col = "dark grey",
  add = FALSE,
  bg = "white",
  diag = TRUE,
  outline = TRUE,
  mar = c(0, 0, 2, 0),
  number.cex = 2,
  cl.pos = "b",
  cl.ratio = 0.125,
  cl.align.text = "l",
  cl.offset = 0.2,
  tl.srt = 45,
  tl.pos = "lt",
  tl.offset = 0.2,
  tl.col = "black",
  pch.col = "white",
  addgrid.col = "black",
  xpd = TRUE,
  tl.cex = 1.3,
  is.corr = TRUE,
  title = "East Asian"
)

## Part 3: Binary (case/control) traits example in European ancestry

In [ ]:
# Files
preview_text(file.path(eur_dir, "SCZ_chr1.txt"))
preview_text(file.path(eur_dir, "BIP_chr1.txt"))

files_bin <- c(
  file.path(eur_dir, "SCZ_chr1.txt"),
  file.path(eur_dir, "BIP_chr1.txt")
)

hm3_bin <- file.path(eur_dir, "eur_w_ld_chr", "w_hm3.snplist")
trait_names_bin <- c("SCZ", "BIP")
N_bin <- c(NA, NA)
info.filter <- 0.9
maf.filter <- 0.01

munge(
  files = files_bin,
  hm3 = hm3_bin,
  trait.names = trait_names_bin,
  N = N_bin,
  info.filter = info.filter,
  maf.filter = maf.filter
)

In [ ]:
preview_gz(file.path(eur_dir, "SCZ.sumstats.gz"))
preview_gz(file.path(eur_dir, "BIP.sumstats.gz"))

traits_bin <- c(
  file.path(eur_dir, "SCZ.sumstats.gz"),
  file.path(eur_dir, "BIP.sumstats.gz")
)

ld_bin <- file.path(eur_dir, "eur_w_ld_chr")
wld_bin <- file.path(eur_dir, "eur_w_ld_chr")
sample.prev_bin <- c(0.5, 0.5)
population.prev_bin <- c(0.01, 0.02)
trait_names_ldsc_bin <- c("SCZ", "BIP")

LDSC_SCZ_BIP <- ldsc(
  traits = traits_bin,
  sample.prev = sample.prev_bin,
  population.prev = population.prev_bin,
  ld = ld_bin,
  wld = wld_bin,
  trait.names = trait_names_ldsc_bin
)

save(LDSC_SCZ_BIP, file = file.path(eur_dir, "LDSC_SCZ_BIP.RData"))

In [ ]:
# Genetic covariance matrix
LDSC_SCZ_BIP$S

# Genetic correlation matrix
cov2cor(LDSC_SCZ_BIP$S)

# Standard errors
k <- nrow(LDSC_SCZ_BIP$S)
SE <- matrix(0, k, k)
SE[lower.tri(SE, diag = TRUE)] <- sqrt(diag(LDSC_SCZ_BIP$V))
SE

# Z-statistics
Z <- LDSC_SCZ_BIP$S / SE
Z

# P-values
P <- 2 * pnorm(Z, lower.tail = FALSE)
P

In [ ]:
rownames(LDSC_SCZ_BIP$S) <- colnames(LDSC_SCZ_BIP$S)

corrplot(
  corr = cov2cor(LDSC_SCZ_BIP$S),
  method = "color",
  addCoef.col = "dark grey",
  add = FALSE,
  bg = "white",
  diag = TRUE,
  outline = TRUE,
  mar = c(0, 0, 2, 0),
  number.cex = 2,
  cl.pos = "b",
  cl.ratio = 0.125,
  cl.align.text = "l",
  cl.offset = 0.2,
  tl.srt = 45,
  tl.pos = "lt",
  tl.offset = 0.2,
  tl.col = "black",
  pch.col = "white",
  addgrid.col = "black",
  xpd = TRUE,
  tl.cex = 1.3,
  is.corr = TRUE,
  title = "SCZ / BIP"
)

## End

You have now run the Colab version of Practical 2 for genetic correlation.